In [ ]:
from scipy.stats import bootstrap
import numpy as np

In [ ]:
# Fixa a semente do NumPy: os dados gerados serão sempre os mesmos
np.random.seed(42)

# Gera 200 valores de uma normal (média=100, desvio padrão=15): a amostra original
dados = np.random.normal(loc=100, scale=15, size=200)

# Número de reamostragens (sorteios com reposição)
n_resamples = 1000

# Executa o bootstrap e guarda o resultado
resultado = bootstrap(
    (dados,),                    # amostra dentro de uma tupla (formato exigido)
    np.mean,                     # estatística calculada em cada reamostragem
    confidence_level=0.95,       # intervalo de confiança de 95%
    n_resamples=n_resamples,     # quantas reamostragens fazer
    method='percentile',         # IC pelos percentis 2,5% e 97,5%
    random_state=42              # semente: reamostragens reproduzíveis
)

# Média da amostra original (deve ficar perto de 100)
print("Média estimada: ", np.mean(dados))

# Intervalo de confiança de 95% da média (limites inferior e superior)
print("IC 95%: ", resultado.confidence_interval)

Média estimada:  99.38843552241873
IC 95%:  ConfidenceInterval(low=np.float64(97.35697378056044), high=np.float64(101.35486142344855))


In [7]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

In [ ]:
# Carrega o dataset Diabetes: X = variáveis dos pacientes (idade, IMC, pressão...),
# y = progressão da doença (o valor que queremos prever)
X, y = load_diabetes(return_X_y=True)

# Cria o modelo de regressão linear
modelo = LinearRegression()

# Função que recebe o número de folds (k) e devolve o erro de cada fold
def avaliar_Kfold(k):
    # Prepara a divisão em k partes; shuffle embaralha antes, random_state=42 deixa reprodutível
    kf = KFold(n_splits=k, shuffle=True, random_state=42)

    # Lista vazia para guardar o erro de cada rodada
    erros = []

    # Repete k vezes; a cada volta recebe os índices das linhas de treino e de teste
    for train_index, test_index in kf.split(X):
        # Usa os índices para separar as variáveis em treino e teste
        X_train, X_test = X[train_index], X[test_index]

        # Separa o alvo (as respostas) do mesmo jeito
        y_train, y_test = y[train_index], y[test_index]

        # Treina o modelo só com os dados de treino (recomeça do zero a cada volta)
        modelo.fit(X_train, y_train)

        # Prevê o alvo dos dados de teste, que o modelo nunca viu
        pred = modelo.predict(X_test)

        # Calcula o MSE: média de (real - previsto)², quanto menor, melhor
        erro = mean_squared_error(y_test, pred)

        # Guarda o erro dessa volta na lista
        erros.append(erro)

    # Devolve os k erros como array NumPy
    return np.array(erros)

# Valores de K que serão comparados
folds = [2,5,10,20]

# Roda o experimento para cada K
for k in folds:
    # Executa a validação cruzada e pega os erros de cada fold
    erros = avaliar_Kfold(k)

    # Mostra qual K está sendo avaliado

K = 2
MSE Médio:  3037.6524665243146
Desvio padrão:  59.05413838859113
----------------------------------------
K = 5
MSE Médio:  3015.3816942872713
Desvio padrão:  291.23979402678174
----------------------------------------
K = 10
MSE Médio:  3023.6568282792787
Desvio padrão:  408.60659088926326
----------------------------------------
K = 20
MSE Médio:  3017.7331683481557
Desvio padrão:  720.3465020665623
----------------------------------------
